# Hoppe Additivity Analysis
### Perovskite and Pyrochlore Screening from MaplePy Outputs

**Purpose:** Compute Hoppe additivity residuals for ternary oxides
processed by MaplePy, classify failures, and generate publication
figures for *J. Appl. Cryst.*

**Relationship to MaplePy:** This notebook is a downstream consumer.
It reads `maple_summary.csv` and `maple_results.csv` produced by
MaplePy’s batch processing (Section 5 or 7), and does not modify
or extend the MaplePy codebase.

**Requirements:** pandas, numpy, matplotlib.  No pymatgen needed
(all crystallographic computation was done upstream by MaplePy).

---

In [ ]:
# !pip install adjustText

## 0. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# ── Sheffield colour palette ───────────────────────────────────────────────────
SHEFFIELD = {
    'brand_violet' : '#440099',
    'dark_violet'  : '#24125E',
    'electric'     : '#7000FF',
    'powder_violet': '#A78CFA',
    'pale_violet'  : '#C8BBFF',
    'teal'         : '#005A8F',
    'aqua'         : '#00BBCC',
    'coral'        : '#E7004C',
    'flamingo'     : '#FF6371',
    'peach'        : '#FF9664',
    'peak_green'   : '#005750',
    'spearmint'    : '#3BD4AE',
    'midnight'     : '#131E29',
    'pastel_green' : '#A1DED2',
    'powder_blue'  : '#9ADBE8',
}

plt.rcParams.update({
    'font.family'      : 'sans-serif',
    'font.size'        : 11,
    'axes.labelsize'   : 13,
    'axes.titlesize'   : 14,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'axes.linewidth'   : 1.0,
    'xtick.direction'  : 'in',
    'ytick.direction'  : 'in',
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
})

print("Hoppe Additivity Analysis loaded.")

In [ ]:
# Uses a tkinter directory picker to select the folder containing
# maple_summary.csv and maple_results.csv from a MaplePy batch run.

import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

MAPLE_DIR = filedialog.askdirectory(
    title="Select folder containing MaplePy output CSVs "
          "(maple_summary.csv and maple_results.csv)"
)
root.destroy()

if not MAPLE_DIR:
    raise FileNotFoundError(
        "No directory selected. Re-run this cell and choose the "
        "folder containing your MaplePy output CSVs."
    )

MAPLE_DIR = Path(MAPLE_DIR)
print(f"Selected directory: {MAPLE_DIR}")

# ── Locate CSVs ──────────────────────────────────────────────────────────
summary_csv = MAPLE_DIR / "maple_summary.csv"
results_csv = MAPLE_DIR / "maple_results.csv"

if not summary_csv.exists():
    # Check for outputs_* subdirectories (MaplePy timestamped folders)
    output_dirs = sorted(MAPLE_DIR.glob('outputs_*'), reverse=True)
    if output_dirs:
        MAPLE_DIR = output_dirs[0]
        summary_csv = MAPLE_DIR / 'maple_summary.csv'
        results_csv = MAPLE_DIR / 'maple_results.csv'
        print(f"  Using most recent output: {MAPLE_DIR.name}")

if not summary_csv.exists():
    raise FileNotFoundError(
        f"maple_summary.csv not found in {MAPLE_DIR}.\n"
        f"Run MaplePy batch processing (Section 5 or 7) first."
    )
if not results_csv.exists():
    raise FileNotFoundError(
        f"maple_results.csv not found in {MAPLE_DIR}.\n"
        f"Run MaplePy batch processing (Section 5 or 7) first."
    )

# ── Load ─────────────────────────────────────────────────────────────────
df_summary = pd.read_csv(summary_csv)
df_results = pd.read_csv(results_csv)

print(f"\nLoaded: {len(df_summary)} structures from {summary_csv.name}")
print(f"        {len(df_results)} site-level rows from {results_csv.name}")
print(f"        {df_summary['compound'].nunique()} unique compositions")
print(f"        {df_summary['space_group'].nunique()} space groups")
print(f"        {df_summary['disordered'].sum()} disordered structures")

## 1. Binary Oxide Reference MAPLE Table

See inline comments for full provenance documentation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#
# Fallback values from MaplePy validation structures and Hoppe (1995).
# Format: {reduced_formula: (maple_kcal_mol, structure_type, source)}
#
# UPDATE these with your own MaplePy batch results on ICSD CIFs.
# The fallback values are a starting point, not a definitive reference.

BINARY_MAPLE_FALLBACK = {

    # ── Rock salt MO (Fm-3m) ──────────────────────────────────────────
    # ICSD median values
    "MgO":(1102.2, "Rock salt Fm-3m", "Median of 34 ambient ICSD Fm-3m entries, range 1092-1106"),
    "CaO":(965.0,  "Rock salt Fm-3m", "Median of 15 ambient ICSD Fm-3m entries, range 964-972"),
    "SrO":(900.0,  "Rock salt Fm-3m", "Median of 11 ambient ICSD Fm-3m entries, range 898-906"),
    "BaO":(838.5,  "Rock salt Fm-3m", "Median of 4 ambient ICSD Fm-3m entries, range 838-842; 1 zero excluded"),
    "MnO":(1044.7, "Rock salt Fm-3m", "Median of 13 ambient ICSD Fm-3m entries, range 1036-1053; 1 zero excluded"),
    "FeO":(1077.4, "Rock salt Fm-3m", "Median of 5 ambient ICSD Fm-3m entries, range 1073-1082; 2 zeros excluded"),
    "CoO":(1090.4, "Rock salt Fm-3m", "Median of 22 ambient ICSD Fm-3m entries, range 1086-1100"),
    "NiO":(1111.1, "Rock salt Fm-3m", "Median of 30 ambient ICSD Fm-3m entries, range 1052-1119"),
    "CdO":(988.6,  "Rock salt Fm-3m", "Median of 14 ambient ICSD Fm-3m entries, range 982-992; 3 zeros excluded"),
    "TiO":(1110.6, "Rock salt Fm-3m", "Median of 8 ambient ICSD Fm-3m entries, range 1081-1113; 2 zeros excluded"),
    "VO":(1136.8,  "Rock salt Fm-3m", "Median of 9 ambient ICSD Fm-3m entries, range 1127-1143; 1 zero excluded"),
    "ZnO":(1086.0, "Rock salt Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 1085-1087; HP polymorph"),
    # Estimated rock salts
    "EuO":(902.9,  "Rock salt Fm-3m", "Median of 9 ambient ICSD Fm-3m entries, range 901-904; 3 zeros excluded"),
    "CuO":(1093.6, "Rock salt Fm-3m", "1 ambient ICSD Fm-3m entry; HP polymorph, not tenorite"),
    "PbO":(810.0,  "Litharge",         "Estimated, to be updated"),

    # ── Antifluorite M2O (Fm-3m) ──────────────────────────────────────
    # ICSD median values
    "Li2O":(837.7,  "Antifluorite Fm-3m", "Median of 10 ambient ICSD Fm-3m entries, range 835-838"),
    "Na2O":(694.5,  "Antifluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 694-695"),
    "K2O":(599.8,   "Antifluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 599-600"),
    "Rb2O":(572.6,  "Antifluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 572-573"),
    # Estimated
    "Cs2O":(450.0,  "Antifluorite Fm-3m", "Estimated, to be updated"),
    "Ag2O":(700.0,  "Cuprite Pn-3m",      "Estimated, to be updated"),
    "Cu2O":(730.0,  "Cuprite Pn-3m",      "MaplePy v4, a=4.270"),
    "Tl2O":(650.0,  "Various",            "Estimated, to be updated"),
    "Pd2O":(700.0,  "Various",            "Estimated, Pd1+ binary"),

    # ── Rutile MO2 (P4_2/mnm) ─────────────────────────────────────────
    # ICSD median values
    "TiO2":(3256.2, "Rutile P4_2/mnm", "Median of 112 ambient ICSD P4_2/mnm entries, range 3218-3505"),
    "SnO2":(3110.2, "Rutile P4_2/mnm", "Median of 48 ambient ICSD P4_2/mnm entries, range 3071-3156"),
    "MnO2":(3382.9, "Rutile P4_2/mnm", "Median of 19 ambient ICSD P4_2/mnm entries, range 3332-3388; 1 zero excluded"),
    "GeO2":(3389.6, "Rutile P4_2/mnm", "Median of 8 ambient ICSD P4_2/mnm entries, range 3386-3425"),
    "RuO2":(3244.8, "Rutile P4_2/mnm", "Median of 17 ambient ICSD P4_2/mnm entries, range 3228-3252"),
    "IrO2":(3225.5, "Rutile P4_2/mnm", "Median of 8 ambient ICSD P4_2/mnm entries, range 3221-3235"),
    "PbO2":(2955.0, "Rutile P4_2/mnm", "Median of 11 ambient ICSD P4_2/mnm entries, range 2946-2959"),
    "VO2":(3291.8,  "Rutile P4_2/mnm", "Median of 5 ambient ICSD P4_2/mnm entries, range 3284-3317"),
    "CrO2":(3352.8, "Rutile P4_2/mnm", "Median of 14 ambient ICSD P4_2/mnm entries, range 3346-3424"),
    "OsO2":(3210.8, "Rutile P4_2/mnm", "Median of 5 ambient ICSD P4_2/mnm entries, range 3205-3215"),
    "NbO2":(3153.4, "Rutile P4_2/mnm", "Median of 3 ambient ICSD P4_2/mnm entries, range 3120-3154"),
    "MoO2":(3179.7, "Rutile P4_2/mnm", "Median of 2 ambient ICSD P4_2/mnm entries, range 3179-3180"),
    "WO2":(3196.9,  "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "RhO2":(3264.4, "Rutile P4_2/mnm", "Median of 2 ambient ICSD P4_2/mnm entries, range 3261-3268; 1 zero excluded"),
    "PtO2":(3239.7, "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "PdO2":(3251.9, "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "ReO2":(3177.5, "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "TaO2":(3145.9, "Rutile P4_2/mnm", "Median of 2 ambient ICSD P4_2/mnm entries, range 3130-3162"),
    "SiO2":(3593.7, "Stishovite P4_2/mnm", "Median of 42 ambient ICSD P4_2/mnm entries, range 3575-3664"),
    # Estimated rutiles
    "TcO2":(3250.0, "Rutile P4_2/mnm", "Estimated from Ru/Rh/Os trend"),
    "SbO2":(3100.0, "Rutile P4_2/mnm", "Estimated, to be updated"),

    # ── Fluorite MO2 (Fm-3m) ──────────────────────────────────────────
    # ICSD median values
    "CeO2":(2856.3, "Fluorite Fm-3m", "Median of 64 ambient ICSD Fm-3m entries, range 2796-2891; 4 zeros excluded"),
    "ZrO2":(3020.0, "Fluorite Fm-3m", "Median of 12 ambient ICSD Fm-3m entries, range 3001-3176"),
    "HfO2":(3034.4, "Fluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 3016-3053"),
    "ThO2":(2761.6, "Fluorite Fm-3m", "Median of 15 ambient ICSD Fm-3m entries, range 2758-2770; 3 zeros excluded"),
    "UO2":(2826.6,  "Fluorite Fm-3m", "Median of 25 ambient ICSD Fm-3m entries, range 2821-2854; 2 zeros excluded"),
    "PrO2":(2865.5, "Fluorite Fm-3m", "Median of 5 ambient ICSD Fm-3m entries, range 2861-2872; 3 zeros excluded"),
    "TbO2":(2965.0, "Fluorite Fm-3m", "1 ambient ICSD Fm-3m entry; 2 zeros excluded"),
    "NpO2":(2843.9, "Fluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 2843-2844; 4 zeros excluded"),
    "AmO2":(2874.0, "Fluorite Fm-3m", "Median of 5 ambient ICSD Fm-3m entries, range 2867-2876"),
    "CmO2":(2879.4, "Fluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 2877-2882; 5 zeros excluded"),
    "PuO2":(2863.9, "Fluorite Fm-3m", "Median of 3 ambient ICSD Fm-3m entries, range 2863-2864; 2 zeros excluded"),

    # ── Corundum M2O3 (R-3c) ──────────────────────────────────────────
    # ICSD median values
    "Al2O3":(4356.2, "Corundum R-3c", "Median of 68 ambient ICSD R-3c entries, range 4250-4369"),
    "Cr2O3":(4170.3, "Corundum R-3c", "Median of 21 ambient ICSD R-3c entries, range 4159-4190"),
    "Fe2O3":(4121.9, "Corundum R-3c", "Median of 53 ambient ICSD R-3c entries, range 4074-4207; 2 outliers excluded"),
    "V2O3":(4124.1,  "Corundum R-3c", "Median of 14 ambient ICSD R-3c entries, range 4119-4924"),
    "Ti2O3":(4036.2, "Corundum R-3c", "Median of 12 ambient ICSD R-3c entries, range 4031-4066"),
    "Ga2O3":(4183.4, "Corundum R-3c", "Median of 3 ambient ICSD R-3c entries, range 4181-4198"),
    "Rh2O3":(4072.3, "Corundum R-3c", "Median of 4 ambient ICSD R-3c entries, range 4024-4087"),
    # Estimated corundums
    "Ni2O3":(4100.0, "Estimated",     "Interpolated from 3d corundum trend Ti-V-Cr-Fe; no stable ambient phase in ICSD"),

    # ── Spinel (Fd-3m) ────────────────────────────────────────────────
    # Co2O3 does not exist as a stable ambient phase; Co3O4 spinel used instead
    "Co3O4":(5445.2, "Spinel Fd-3m", "Median of 26 ambient ICSD Fd-3m entries, range 5020-5508"),

    # ── Bixbyite M2O3 (Ia-3) ─────────────────────────────────────────
    # ICSD median values
    "Mn2O3":(4133.9, "Bixbyite Ia-3", "Median of 10 ambient ICSD Ia-3 entries, range 4087-4155"),
    "Y2O3":(3652.0,  "Bixbyite Ia-3", "Median of 38 ambient ICSD Ia-3 entries, range 3585-4179; 2 zeros excluded"),
    "Sc2O3":(3929.6, "Bixbyite Ia-3", "Median of 8 ambient ICSD Ia-3 entries, range 3927-3966"),
    "Tl2O3":(3675.2, "Bixbyite Ia-3", "Median of 4 ambient ICSD Ia-3 entries, range 3663-3700"),
    "In2O3":(3822.4, "Bixbyite Ia-3", "Median of 18 ambient ICSD Ia-3 entries, range 3793-3849"),

    # ── RE sesquioxides ───────────────────────────────────────────────
    # ICSD median values — A-type (P-3m1) for La; bixbyite (Ia-3) for rest
    "La2O3":(3390.9, "A-type P-3m1",  "Median of 11 ambient ICSD P-3m1 entries, range 3337-3436"),
    "Ce2O3":(3483.6, "Bixbyite Ia-3", "1 ambient ICSD Ia-3 entry"),
    "Pr2O3":(3475.6, "Bixbyite Ia-3", "Median of 2 ambient ICSD Ia-3 entries, range 3476-3476"),
    "Nd2O3":(3496.8, "Bixbyite Ia-3", "Median of 3 ambient ICSD Ia-3 entries, range 3495-3507"),
    "Pm2O3":(3525.4, "Bixbyite Ia-3", "1 ambient ICSD Ia-3 entry"),
    "Sm2O3":(3545.1, "Bixbyite Ia-3", "Median of 5 ambient ICSD Ia-3 entries, range 3541-3558"),
    "Eu2O3":(3569.8, "Bixbyite Ia-3", "Median of 6 ambient ICSD Ia-3 entries, range 3559-3630"),
    "Gd2O3":(3581.4, "Bixbyite Ia-3", "Median of 14 ambient ICSD Ia-3 entries, range 3573-3611"),
    "Tb2O3":(3608.5, "Bixbyite Ia-3", "Median of 5 ambient ICSD Ia-3 entries, range 3592-3625"),
    "Dy2O3":(3631.2, "Bixbyite Ia-3", "Median of 10 ambient ICSD Ia-3 entries, range 3624-3646"),
    "Ho2O3":(3650.6, "Bixbyite Ia-3", "Median of 9 ambient ICSD Ia-3 entries, range 3641-3655"),
    "Er2O3":(3670.2, "Bixbyite Ia-3", "Median of 10 ambient ICSD Ia-3 entries, range 3655-3689"),
    "Tm2O3":(3698.6, "Bixbyite Ia-3", "Median of 5 ambient ICSD Ia-3 entries, range 3691-3710"),
    "Yb2O3":(3712.0, "Bixbyite Ia-3", "Median of 17 ambient ICSD Ia-3 entries, range 3707-3724"),
    "Lu2O3":(3727.0, "Bixbyite Ia-3", "Median of 12 ambient ICSD Ia-3 entries, range 3703-3735"),

    # ── Other binaries ────────────────────────────────────────────────
    "Bi2O3":(2580.0, "Monoclinic P2_1/c", "MaplePy v4, alpha-Bi2O3 — to be updated"),
    "HgO":(850.0,    "Various",            "Estimated, to be updated"),

    # ── Higher-valent oxides ──────────────────────────────────────────
    "V2O5":(9454.0,  "Pmmn",  "Median of 17 ambient ICSD Pmmn entries, range 9393-9638"),
    "Nb2O5":(8988.2, "C2/c",  "Median of 2 ambient ICSD C2/c entries, range 8984-8992"),
    "Ta2O5":(9127.2, "C2/c",  "1 ambient ICSD C2/c entry"),
    "Sb2O5":(8955.4, "C2/c",  "Median of 2 ambient ICSD C2/c entries, range 8955-8956"),
    "Bi2O5":(4300.0, "Various",     "Estimated, to be updated"),
    "WO3":(4600.0,   "ReO3-type",   "Estimated, to be updated"),
    "MoO3":(4500.0,  "Various",     "Estimated, to be updated"),
    "CrO3":(4400.0,  "Various",     "Estimated, to be updated"),
    "OsO3":(4500.0,  "Various",     "Estimated, to be updated"),
    "RuO3":(4500.0,  "Various",     "Estimated, Ru6+ binary"),
    "ReO3":(4600.0,  "ReO3-type",   "Estimated, Re6+ binary"),
    "Re2O5":(4400.0, "Various",     "Estimated, Re5+ binary"),
    "Re2O7":(5000.0, "Various",     "Estimated, to be updated"),
    "Tc2O7":(5000.0, "Various",     "Estimated, to be updated"),
    "Ru2O5":(4250.0, "Various",     "Estimated, Ru5+ binary"),
    "W2O5":(4300.0,  "Various",     "Estimated, W5+ binary"),
    "Mo2O3":(3300.0, "Various",     "Estimated, Mo3+ binary"),
}


def load_binary_table(csv_path=None, fallback=True):
    """
    Load the binary oxide MAPLE reference table.

    Priority:
    1. CSV from a MaplePy batch run on binary CIFs (if csv_path given)
    2. Fallback dict above

    Returns
    -------
    dict  {reduced_formula: maple_kcal_mol}
    pd.DataFrame  for display
    """
    lookup = {}
    rows = []

    if csv_path is not None and Path(csv_path).exists():
        df = pd.read_csv(csv_path)
        print(f"Loaded binary reference from {csv_path}")
        name_col = 'compound' if 'compound' in df.columns else 'formula'
        for _, r in df.iterrows():
            formula = r[name_col]
            maple = r['total_maple'] if 'total_maple' in df.columns \
                    else r.get('maple_kcal_mol')
            if pd.notna(maple):
                lookup[formula] = float(maple)
                rows.append({"formula": formula,
                             "maple_kcal_mol": float(maple),
                             "source": "ICSD CIF batch"})
        print(f"  {len(lookup)} binaries loaded from CSV.")

    if fallback:
        n_added = 0
        for formula, (maple, stype, source) in BINARY_MAPLE_FALLBACK.items():
            if formula not in lookup:
                lookup[formula] = maple
                rows.append({"formula": formula,
                             "maple_kcal_mol": maple,
                             "source": source})
                n_added += 1
        if n_added > 0:
            print(f"  {n_added} binaries added from fallback table.")

    df_out = pd.DataFrame(rows).sort_values("formula").reset_index(drop=True)
    print(f"  Total binary references: {len(lookup)}")
    return lookup, df_out


# ── Load the table ──────────────────────────────────────────────────────────
# To use your own ICSD batch results, set BINARY_CSV below.
BINARY_CSV = None   # e.g. Path("binary_oxides/maple_summary.csv")

BINARY_MAPLE, df_binary = load_binary_table(csv_path=BINARY_CSV)
display(df_binary)

## 2. Ternary Decomposition Function

For an oxide A$_x$B$_y$O$_u$ with known cation oxidation states,
the decomposition into binary constituents is determined by
stoichiometry and charge balance.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

def _binary_formula_for(element, oxi_state):
    """
    Return the expected binary oxide formula for a cation at a
    given oxidation state, plus the cation and oxygen counts
    per formula unit of that binary.

    Returns (formula_str, n_cations, n_oxygens) or (None, None, None).
    """
    oxi = round(abs(oxi_state))

    # ── Special cases ─────────────────────────────────────────────
    # Co3O4 spinel: Co2+[Co3+]2O4. Use as the Co3+ reference.
    # Decomposition: 1 Co3+ requires 1/3 Co3O4 (3 Co per fu),
    # contributing 4/3 O per Co3+.
    if element == "Co" and oxi == 3:
        return ("Co3O4", 3, 4)

    patterns = {
        1: (f"{element}2O",  2, 1),
        2: (f"{element}O",   1, 1),
        3: (f"{element}2O3", 2, 3),
        4: (f"{element}O2",  1, 2),
        5: (f"{element}2O5", 2, 5),
        6: (f"{element}O3",  1, 3),
        7: (f"{element}2O7", 2, 7),
    }
    return patterns.get(oxi, (None, None, None))


def decompose_ternary(source_cif, df_results, binary_lookup, verbose=True):
    """
    Decompose a ternary oxide into binary components using the
    per-site oxidation states from MaplePy output.

    Parameters
    ----------
    source_cif     : str   filename from maple_summary/results
    df_results     : DataFrame  site-level MaplePy output
    binary_lookup  : dict  {formula: maple_kcal_mol}
    verbose        : bool

    Returns
    -------
    dict with keys:
        source_cif, compound, maple_ternary,
        decomposition (list of dicts per cation element),
        maple_binary_sum, delta_abs, delta_pct,
        oxygen_balanced, missing_binaries, success
    """
    sites = df_results[df_results['source_cif'] == source_cif].copy()

    if sites.empty:
        return {"source_cif": source_cif, "success": False,
                "error": "No site data found"}

    compound     = sites['compound'].iloc[0]
    maple_tern   = sites['total_maple'].iloc[0]
    Z            = sites['Z'].iloc[0]

    # ── Identify cations and their mean oxidation states ──────────────
    cation_sites = sites[sites['oxidation_state'] > 0]
    anion_sites  = sites[sites['oxidation_state'] < 0]

    if cation_sites.empty:
        return {"source_cif": source_cif, "compound": compound,
                "success": False, "error": "No cation sites identified"}

    cation_info = {}
    for el in cation_sites['element'].unique():
        el_sites = cation_sites[cation_sites['element'] == el]
        cation_info[el] = {
            'count_per_fu': len(el_sites) / Z,
            'mean_oxi': el_sites['oxidation_state'].mean(),
            'sites_in_cell': len(el_sites),
        }

    o_per_fu = len(anion_sites) / Z

    # ── Decompose each cation into its binary ─────────────────────────
    decomposition = []
    maple_sum     = 0.0
    o_from_binary = 0.0
    missing       = []

    for el, info in cation_info.items():
        binary_formula, n_cat, n_oxy = _binary_formula_for(
            el, info['mean_oxi']
        )

        if binary_formula is None:
            missing.append(f"{el}{info['mean_oxi']:+.1f} (no pattern)")
            continue

        maple_binary = binary_lookup.get(binary_formula)
        if maple_binary is None:
            missing.append(f"{binary_formula} (not in table)")
            continue

        coeff = info['count_per_fu'] / n_cat
        contribution = coeff * maple_binary
        o_contrib    = coeff * n_oxy

        decomposition.append({
            'element'      : el,
            'oxi_state'    : round(info['mean_oxi'], 2),
            'binary'       : binary_formula,
            'n_cat_binary' : n_cat,
            'n_oxy_binary' : n_oxy,
            'coeff'        : round(coeff, 4),
            'maple_binary' : round(maple_binary, 2),
            'contribution' : round(contribution, 2),
            'o_from_binary': round(o_contrib, 4),
        })

        maple_sum     += contribution
        o_from_binary += o_contrib

    # ── Oxygen balance check ──────────────────────────────────────────
    o_residual = abs(o_from_binary - o_per_fu)
    o_balanced = o_residual < 0.05

    # ── Additivity residual ───────────────────────────────────────────
    delta_abs = abs(maple_tern - maple_sum) if decomposition else None
    delta_pct = (100 * delta_abs / maple_tern
                 if delta_abs is not None and maple_tern > 0
                 else None)

    result = {
        'source_cif'       : source_cif,
        'compound'         : compound,
        'maple_ternary'    : round(maple_tern, 2),
        'Z'                : Z,
        'n_cation_elements': len(cation_info),
        'decomposition'    : decomposition,
        'maple_binary_sum' : round(maple_sum, 2),
        'delta_abs'        : round(delta_abs, 2) if delta_abs else None,
        'delta_pct'        : round(delta_pct, 3) if delta_pct else None,
        'oxygen_per_fu'    : round(o_per_fu, 4),
        'oxygen_from_bins' : round(o_from_binary, 4),
        'oxygen_balanced'  : o_balanced,
        'missing_binaries' : missing,
        'success'          : len(decomposition) > 0 and len(missing) == 0,
    }

    if verbose and result['success']:
        status = ("GOOD (<1%)" if delta_pct < 1.0 else
                  "MARGINAL (1-2%)" if delta_pct < 2.0 else
                  "POOR (>2%)")
        print(f"  {compound:20s}  "
              f"MAPLE={maple_tern:8.1f}  "
              f"sum={maple_sum:8.1f}  "
              f"delta={delta_pct:.2f}%  {status}"
              f"{'  [O imbalanced]' if not o_balanced else ''}")
    elif verbose and not result['success']:
        print(f"  {compound:20s}  INCOMPLETE: missing {missing}")

    return result


print("Decomposition function defined.")

## 3. Batch Additivity Calculation

Run the decomposition across all structures in the MaplePy output
and assemble results into a summary DataFrame.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

def batch_additivity(df_summary, df_results, binary_lookup,
                     verbose=False):
    """
    Compute additivity residuals for all structures.

    Parameters
    ----------
    df_summary    : DataFrame from maple_summary.csv
    df_results    : DataFrame from maple_results.csv
    binary_lookup : dict {formula: maple_kcal_mol}
    verbose       : bool

    Returns
    -------
    pd.DataFrame  one row per structure with additivity results
    list of dicts  full decomposition details (for inspection)
    """
    rows = []
    details = []

    for _, summary_row in df_summary.iterrows():
        cif = summary_row['source_cif']

        result = decompose_ternary(
            cif, df_results, binary_lookup, verbose=verbose
        )
        details.append(result)

        rows.append({
            'source_cif'       : cif,
            'compound'         : result.get('compound', ''),
            'space_group'      : summary_row['space_group'],
            'Z'                : result.get('Z'),
            'disordered'       : summary_row['disordered'],
            'maple_ternary'    : result.get('maple_ternary'),
            'maple_binary_sum' : result.get('maple_binary_sum'),
            'delta_abs'        : result.get('delta_abs'),
            'delta_pct'        : result.get('delta_pct'),
            'madelung_factor'  : summary_row['madelung_factor'],
            'd_ref_ang'        : summary_row['d_ref_ang'],
            'oxygen_balanced'  : result.get('oxygen_balanced'),
            'n_cation_elements': result.get('n_cation_elements'),
            'missing_binaries' : '; '.join(result.get('missing_binaries', [])),
            'success'          : result.get('success', False),
        })

    df = pd.DataFrame(rows)

    # Summary statistics
    n_total   = len(df)
    n_success = df['success'].sum()
    n_fail    = n_total - n_success

    if n_success > 0:
        good     = (df['success'] & (df['delta_pct'] < 1.0)).sum()
        marginal = (df['success'] & (df['delta_pct'] >= 1.0)
                    & (df['delta_pct'] < 2.0)).sum()
        poor     = (df['success'] & (df['delta_pct'] >= 2.0)).sum()
    else:
        good = marginal = poor = 0

    print(f"\n{'='*65}")
    print(f"  Additivity analysis: {n_total} structures")
    print(f"  Decomposed successfully: {n_success}")
    print(f"    Good (<1%):     {good}")
    print(f"    Marginal (1-2%): {marginal}")
    print(f"    Poor (>2%):     {poor}")
    print(f"  Incomplete (missing binaries): {n_fail}")
    print(f"{'='*65}")

    return df, details


# ── Run the batch ────────────────────────────────────────────────────────────
print("Computing additivity residuals for all structures...")
df_additivity, additivity_details = batch_additivity(
    df_summary, df_results, BINARY_MAPLE, verbose=False
)

# Show the successful decompositions
df_ok = df_additivity[df_additivity['success']].sort_values('delta_pct')
print(f"\nSuccessfully decomposed: {len(df_ok)} structures")
if len(df_ok) > 0:
    display(df_ok[['compound','space_group','Z','maple_ternary',
                    'maple_binary_sum','delta_abs','delta_pct',
                    'oxygen_balanced','disordered']].head(30))

## 4. Failure Classification — Tiered Numeric Scheme

| Code | Tier | Label |
|------|------|-------|
| Additive | 0 | Additive (Δ < 1%) |
| 1.1 | 1 | Missing binary reference |
| 2.1 | 2 | Oxidation state assignment error |
| 2.2 | 2 | Z = 1 metadata artefact |
| 2.3 | 2 | Disorder / mean-field |
| 2.4 | 2 | Oxygen non-stoichiometry |
| 3.1 | 3 | Genuine violation |

**Priority:** Additive first, then 1.1 → 2.1 → 2.2 → 2.3 → 2.4 → 3.1.

**Reserved:** 1.2 (upstream MaplePy failure), 2.5 (wrong structure type), 2.6 (anion vacancy disorder).

In [ ]:
# 4. Failure Classification — Tiered Numeric Scheme
# ─────────────────────────────────────────────────────────────────────────────
#
# Every structure receives a classification:
#
#   Additive         Δ < 1%, no failure code assigned
#   Tier 1 (1.x)     Calculation incomplete
#   Tier 2 (2.x)     Calculation complete, result compromised
#   Tier 3 (3.x)     Calculation complete, result trustworthy
#
# Priority: additive checked first, then codes applied in ascending
# order (1.1 → 2.1 → 2.2 → 2.3 → 2.4 → 3.1).  A structure triggering
# multiple flags receives the lowest applicable code.
#
# Future codes (reserved, no current data triggers them):
#   1.2  Upstream MaplePy failure (CIF never reached Hopper)
#   2.5  Wrong structure type (incorrect space group / structure type)
#   2.6  Anion vacancy disorder (O vacancies distinct from cation disorder)

# ── Classification registry ──────────────────────────────────────────────
# Central definition of all codes.  Add new codes here; the classifier
# and all downstream plotting/summary code reads from this dict.

CLASSIFICATION_REGISTRY = {
    # code : (tier, label, description)
    '1.1': (1, 'Missing binary reference',
            'One or more binary oxide components absent from the reference '
            'table. Residual cannot be computed.'),
    '1.2': (1, 'Upstream MaplePy failure',
            'Reserved. MaplePy could not process the CIF (appears in '
            'maple_failed.csv, never reaches Hopper).'),
    '2.1': (2, 'Oxidation state assignment error',
            'Delta > 10% in an ordered, oxygen-balanced structure. Points to '
            'incorrect oxidation states in the CIF.'),
    '2.2': (2, 'Z = 1 metadata artefact',
            'pymatgen compositional reduction yields Z = 1, inflating MAPLE '
            'per formula unit.'),
    '2.3': (2, 'Disorder / mean-field',
            'Mixed-occupancy sites processed via mean-field charge '
            'approximation. Residual may reflect averaging artefact.'),
    '2.4': (2, 'Oxygen non-stoichiometry',
            'Oxygen count from binary decomposition does not match ternary. '
            'Decomposition not well-defined (e.g. Co3O4 for Co2O3).'),
    '2.5': (2, 'Structural coordinate error',
            'O-site MAPLE coefficient of variation exceeds 0.05, indicating '
            'unphysical interatomic distances from Wyckoff site mis-assignment '
            'or wrong fractional coordinates. 32 entries from >= 10 publications.'),
    '2.6': (2, 'Anion vacancy disorder',
            'Reserved. Oxygen vacancies distinct from cation disorder (2.3). '
            'Not currently distinguished.'),
    '3.1': (3, 'Genuine violation',
            'Delta >= 1% with correct oxidation states, correct Z, balanced '
            'oxygen, no disorder. Reflects genuine structural chemistry.'),
}

# Convenience lookups
def _tier_for_code(code):
    """Return the tier (1, 2, or 3) for a classification code."""
    return CLASSIFICATION_REGISTRY[code][0]

def _label_for_code(code):
    """Return the human-readable label for a classification code."""
    return CLASSIFICATION_REGISTRY[code][1]

"""
Code 2.5 detector: Wyckoff 4a/4b site mis-assignment in Pnma perovskites.

Detects ICSD entries where the B-site cation is deposited at Wyckoff 4a
instead of the correct 4b in GdFeO3-type perovskites (Pnma, Z=4, 20 sites).

Validated against 938 Pnma perovskites: catches 19 confirmed errors from
at least 4 independent publications, zero false positives.

Usage in the Hopper classification cell:
    
    wyckoff_error = detect_wyckoff_site_error(
        row['source_cif'], results_df, summary_df
    )
    if wyckoff_error:
        code = '2.5'
        label = 'Wyckoff site mis-assignment (B at 4a instead of 4b)'
"""


def detect_structural_coordinate_error(cif_name, results_df, summary_df,
                                        cv_threshold=0.05):
    """
    Detect structural coordinate errors in Pnma perovskites using the
    coefficient of variation (CV) of per-site oxygen MAPLE values.
 
    In a correctly refined GdFeO3-type perovskite (Pnma, Z=4, 20 sites),
    the 12 oxygen sites occupy two crystallographically distinct positions
    (Wyckoff 4c and 8d) but experience similar electrostatic environments,
    giving per-site MAPLE values within ~5% of each other (CV < 0.045).
 
    Structural coordinate errors distort the local coordination geometry
    and produce anomalously different MAPLE values between O1 and O2
    sublattices, pushing CV above 0.05.
 
    Parameters
    ----------
    cif_name : str
        Source CIF filename (must match a row in summary_df).
    results_df : pd.DataFrame
        Per-site MAPLE results with columns: source_cif, element,
        maple_kcal_mol.
    summary_df : pd.DataFrame
        Per-structure summary with columns: source_cif, Z, space_group.
    cv_threshold : float
        CV threshold above which the structure is flagged.
        Default 0.05 (validated gap: 0.045-0.050).
 
    Returns
    -------
    bool or None
        True  — structural coordinate error detected (classify as 2.5).
        False — structure is clean.
        None  — structure is not a testable Pnma perovskite.
    """
    srow = summary_df[summary_df['source_cif'] == cif_name]
    if len(srow) == 0:
        return None
    srow = srow.iloc[0]
 
    # Only test Z=4 Pnma perovskites with 20 sites
    if srow['Z'] != 4 or srow['space_group'] != 'Pnma':
        return None
 
    entry = results_df[results_df['source_cif'] == cif_name]
    if len(entry) != 20:
        return None
 
    oxygens = entry[entry['element'] == 'O']
    if len(oxygens) != 12:
        return None
 
    # Coefficient of variation of per-site oxygen MAPLE
    o_maples = oxygens['maple_kcal_mol'].values.astype(float)
    o_mean = np.mean(o_maples)
 
    if o_mean <= 0:
        return None
 
    cv = float(np.std(o_maples) / o_mean)
 
    # Explicit bool() to avoid numpy.bool_ identity issues
    return bool(cv > cv_threshold)

def classify_additivity(df_add, threshold_additive=1.0):
    """
    Assign a tiered numeric classification to each structure based on
    its additivity residual and metadata.

    Parameters
    ----------
    df_add : DataFrame
        Output from batch_additivity(), one row per structure.
    threshold_additive : float
        Residual threshold (%) below which a structure is classified
        as additive.  Default 1.0 (Hoppe criterion).

    Returns
    -------
    DataFrame with added columns:
        classification       str   'Additive' or code ('1.1', '2.1', etc.)
        classification_tier  int   0 for additive, 1/2/3 for failures
        classification_label str   human-readable label

    Notes
    -----
    Priority order follows ascending code number.  A structure that
    could trigger both 2.1 and 2.3 receives 2.1 (the lower code).
    Additive status is checked first: delta < threshold -> additive, stop.
    """
    df = df_add.copy()
    df['classification']       = ''
    df['classification_tier']  = 0
    df['classification_label'] = ''

    for idx, row in df.iterrows():

        # ── Code 1.1: missing binary reference ──────────────────────
        # Decomposition could not complete because one or more binary
        # oxide components are absent from the reference table.
        if not row['success'] and row['missing_binaries']:
            df.at[idx, 'classification']       = '1.1'
            df.at[idx, 'classification_tier']   = 1
            df.at[idx, 'classification_label']  = _label_for_code('1.1')
            continue

        # Catch-all for other decomposition failures (should not occur
        # with current data, but defensive)
        if not row['success']:
            df.at[idx, 'classification']       = '1.1'
            df.at[idx, 'classification_tier']   = 1
            df.at[idx, 'classification_label']  = 'Decomposition failed'
            continue

        # ── Guard: success=True but delta_pct unavailable ───────────
        # Can occur when maple_ternary == maple_binary_sum exactly,
        # producing delta_abs = 0.0 that becomes NaN through a
        # rounding or serialisation artefact.  Treat as additive.
        if row['delta_pct'] is None or (isinstance(row['delta_pct'], float)
                                         and row['delta_pct'] != row['delta_pct']):
            df.at[idx, 'classification']       = 'Additive'
            df.at[idx, 'classification_tier']   = 0
            df.at[idx, 'classification_label']  = 'Additive'
            continue

        # ── Code 2.5: Wyckoff site mis-assignment ──────────────────
        # B-site at 4a instead of 4b in Pnma perovskites. Must be
        # checked before Additive because DyCrO3 138222 passes the 1%
        # threshold by coincidence despite having the structural error.
        if detect_structural_coordinate_error(row['source_cif'], df_results, df_summary):
            df.at[idx, 'classification']       = '2.5'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.5')
            continue
        
        # ── Additive: delta < threshold ─────────────────────────────
        # Check additive status before entering the failure logic.
        # Structures within Hoppe range get no failure code.
        if (row['delta_pct'] is not None
                and row['delta_pct'] < threshold_additive):
            df.at[idx, 'classification']       = 'Additive'
            df.at[idx, 'classification_tier']   = 0
            df.at[idx, 'classification_label']  = 'Additive'
            continue

        # ── From here, delta >= threshold: assign failure codes ─────
        # Applied in ascending code order so the lowest applicable
        # code wins.

        # ── Code 2.1: oxidation state assignment error ──────────────
        # Very large residual (>10%) in an ordered structure with
        # balanced oxygen points to incorrect oxidation states in the
        # CIF, producing a wrong binary decomposition.
        if (row['delta_pct'] is not None and row['delta_pct'] > 10.0
                and not row['disordered'] and row['oxygen_balanced']):
            df.at[idx, 'classification']       = '2.1'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.1')
            continue

        # ── Code 2.2: Z = 1 metadata artefact ──────────────────────
        # pymatgen compositional reduction gives Z = 1, inflating
        # MAPLE per formula unit.  All Z = 1 structures are flagged
        # conservatively; genuine Z = 1 cases are rare in ICSD data.
        if row['Z'] == 1:
            df.at[idx, 'classification']       = '2.2'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.2')
            continue

        # ── Code 2.3: disorder / mean-field approximation ───────────
        # Structure has mixed-occupancy sites processed via mean-field
        # charge averaging.  The residual may partly reflect the
        # averaging artefact rather than genuine structural chemistry.
        if row['disordered']:
            df.at[idx, 'classification']       = '2.3'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.3')
            continue

        # ── Code 2.4: oxygen non-stoichiometry ──────────────────────
        # Oxygen count from binary decomposition does not match the
        # ternary oxygen count.  Principal cause: Co3O4 used in place
        # of the non-existent Co2O3.
        if not row['oxygen_balanced']:
            df.at[idx, 'classification']       = '2.4'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.4')
            continue

        # ── Code 3.1: genuine violation ─────────────────────────────
        # delta >= threshold with correct oxidation states, correct Z,
        # balanced oxygen, no disorder.  The residual reflects genuine
        # structural chemistry (covalency, strain, etc.).
        if row['delta_pct'] is not None and row['delta_pct'] >= threshold_additive:
            df.at[idx, 'classification']       = '3.1'
            df.at[idx, 'classification_tier']   = 3
            df.at[idx, 'classification_label']  = _label_for_code('3.1')

    return df


# ── Classify ─────────────────────────────────────────────────────────────────
df_classified = classify_additivity(df_additivity)

# ── Summary by tier ──────────────────────────────────────────────────────────
print("Classification summary")
print("=" * 65)

# Tier-level summary
tier_labels = {0: 'Additive', 1: 'Tier 1 (incomplete)',
               2: 'Tier 2 (compromised)', 3: 'Tier 3 (trustworthy)'}
for tier_num in [0, 1, 2, 3]:
    count = (df_classified['classification_tier'] == tier_num).sum()
    if count > 0:
        print(f"  {tier_labels[tier_num]:30s}  {count:5d}")
print()

# Code-level summary
print("Breakdown by classification code:")
for code in ['Additive', '1.1', '2.1', '2.2', '2.3', '2.4', '2.5', '3.1']:
    subset = df_classified[df_classified['classification'] == code]
    if len(subset) > 0:
        label = subset['classification_label'].iloc[0]
        print(f"  {code:10s}  {label:40s}  {len(subset):5d}")
print("=" * 65)
print()

# ── Show each classification group ──────────────────────────────────────────
for code in ['Additive', '1.1', '2.1', '2.2', '2.3', '2.4', '2.5', '3.1']:
    subset = df_classified[df_classified['classification'] == code]
    if len(subset) > 0:
        label = subset['classification_label'].iloc[0]
        tier  = subset['classification_tier'].iloc[0]
        tier_str = f"Tier {tier}" if tier > 0 else "Additive"
        print(f"\n{code} — {label} [{tier_str}] ({len(subset)} structures)")
        cols = ['compound', 'space_group', 'Z', 'delta_pct',
                'disordered', 'oxygen_balanced']
        if code == '1.1':
            cols.append('missing_binaries')
        display(subset[cols].head(15))


## 5. Additivity Residual Distribution (Key Paper Figure)

Histogram of Δ% for successfully decomposed structures,
with Hoppe’s ±1% range marked.  Coloured by classification code.

In [ ]:
# 5a. Additivity Residual Histogram (Key Paper Figure)
# ─────────────────────────────────────────────────────────────────────────────
#
# Two-panel figure:
#   (a) 0-5% at 0.25% bins — shows the bulk of the data
#   (b) >5% at 2.5% bins   — shows the failure tail in detail
#
# Colour scheme (Sheffield brand palette):
#   Additive      pastel_green   (light, recessive — the expected result)
#   3.1 Genuine   teal           (stands out against the green)
#   2.4 O-nonstoich flamingo     (warm pink)
#   2.2 Z=1       peach          (warm orange)
#   2.1 Oxi state coral          (warm red)

from matplotlib.lines import Line2D

# ── Sheffield-palette colour map for classification codes ────────────────
CLASSIFICATION_COLOURS = {
    'Additive': SHEFFIELD['pastel_green'],
    '1.1':      SHEFFIELD['pale_violet'],
    '2.1':      SHEFFIELD['coral'],
    '2.2':      SHEFFIELD['peach'],
    '2.3':      SHEFFIELD['powder_violet'],
    '2.4':      SHEFFIELD['flamingo'],
    '2.5':      SHEFFIELD['brand_violet'],
    '3.1':      SHEFFIELD['teal'],
}

CLASSIFICATION_LABELS = {
    'Additive': r'Additive ($\Delta$ < 1%)',
    '2.1':      '2.1 Oxi state error',
    '2.2':      '2.2 Z = 1 artefact',
    '2.4':      '2.4 O non-stoichiometry',
    '2.5':      '2.5 Wyckoff site error',
    '3.1':      '3.1 Genuine violation',
}

# Marker shapes per tier for greyscale accessibility
TIER_MARKERS = {0: 'o', 1: 'v', 2: 's', 3: 'D'}


def plot_additivity_histogram(df_classified, save_path=None):
    """
    Two-panel histogram of additivity residuals.

    Panel (a): 0-5% range at 0.25% bins with Hoppe 1% threshold.
    Panel (b): >5% tail at 2.5% bins showing failure breakdown.

    Coloured by classification code using the Sheffield brand palette.
    Lighter pastel_green for additive structures so the failure codes
    (warm colours + teal) stand out.
    """
    df_plot = df_classified[df_classified['delta_pct'].notna()].copy()

    if len(df_plot) == 0:
        print("No structures with computed residuals to plot.")
        return

    fig, (ax_main, ax_tail) = plt.subplots(
        1, 2, figsize=(12, 5),
        gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.25}
    )

    # Stacking order: additive at the back, then by code
    code_order = ['Additive', '3.1', '2.4', '2.5', '2.2', '2.1']
    codes_present = [c for c in code_order
                     if c in df_plot['classification'].values]
    colours = [CLASSIFICATION_COLOURS[c] for c in codes_present]
    labels = [CLASSIFICATION_LABELS[c] for c in codes_present]

    # ── Panel (a): 0 to 5% ──────────────────────────────────────────
    bins_main = np.arange(0, 5.25, 0.25)
    df_main = df_plot[df_plot['delta_pct'] <= 5.0]

    data_main = [df_main[df_main['classification'] == c]['delta_pct'].values
                 for c in codes_present]

    ax_main.hist(data_main, bins=bins_main, color=colours, label=labels,
                 edgecolor='white', linewidth=0.5, stacked=True)

    # Hoppe 1% threshold
    ax_main.axvline(x=1.0, color=SHEFFIELD['coral'], linestyle='--',
                    linewidth=1.5, alpha=0.8, zorder=5)
    ax_main.text(1.12, ax_main.get_ylim()[1] * 0.95, "Hoppe 1%",
                 fontsize=10, color=SHEFFIELD['coral'], va='top')

    ax_main.set_xlabel(r'Additivity residual, $\Delta$ / %')
    ax_main.set_ylabel('Count')
    ax_main.set_title('(a)', loc='left', fontweight='bold')
    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)
    ax_main.set_xlim(-0.1, 5.2)
    ax_main.legend(frameon=False, fontsize=9, loc='upper right')

    # Annotation: fraction within 1%
    n_within = (df_plot['delta_pct'] < 1.0).sum()
    n_total = len(df_plot)
    ax_main.text(0.97, 0.12,
                 f'{n_within}/{n_total} ({100*n_within/n_total:.0f}%)\n'
                 r'within $\pm$1%',
                 transform=ax_main.transAxes, ha='right', va='bottom',
                 fontsize=10, color=SHEFFIELD['dark_violet'], style='italic')

    # ── Panel (b): > 5%, uniform 2.5% bins ──────────────────────────
    df_tail = df_plot[df_plot['delta_pct'] > 5.0]
    bins_tail = np.arange(5, 100, 2.5)

    codes_tail = [c for c in code_order
                  if c in df_tail['classification'].values]
    data_tail = [df_tail[df_tail['classification'] == c]['delta_pct'].values
                 for c in codes_tail]
    colours_tail = [CLASSIFICATION_COLOURS[c] for c in codes_tail]

    ax_tail.hist(data_tail, bins=bins_tail, color=colours_tail,
                 edgecolor='white', linewidth=0.5, stacked=True)

    ax_tail.set_xlabel(r'Additivity residual, $\Delta$ / %')
    ax_tail.set_ylabel('Count')
    ax_tail.set_title(r'(b) $\Delta$ > 5%' + f'  (n = {len(df_tail)})',
                       loc='left', fontweight='bold')
    ax_tail.spines['top'].set_visible(False)
    ax_tail.spines['right'].set_visible(False)
    ax_tail.set_xlim(5, 100)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()
    return fig


# ── Plot ─────────────────────────────────────────────────────────────────────
fig_hist = plot_additivity_histogram(
    df_classified,
    save_path=MAPLE_DIR / 'fig1_additivity_histogram.png'
)

In [ ]:
# 5b. Madelung Factor Clustering (Key Paper Figure)
# ─────────────────────────────────────────────────────────────────────────────
#
# Two-panel figure:
#   (a) All structures — overview with labelled common phases
#   (b) Z=1 cluster detail — auto-detected zoom window with ICSD codes
#
# Panel (b) window is auto-detected from the gap in the MF distribution
# between the main perovskite band and the Z=1 cluster.  Override by
# setting the PANEL_B_* variables below to explicit values.
#
# Label strategy:
#   Panel (a): common phases (n >= 5) at median positions
#   Panel (b): clustered compositions (n >= 3) with entry counts,
#              edge singletons/pairs with compound + ICSD CollCode
#
# Marker shapes per tier for greyscale accessibility:
#   Additive (Tier 0): circles
#   Tier 2 (compromised): squares
#   Tier 3 (genuine): diamonds

from adjustText import adjust_text
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# ── User-configurable panel (b) window ───────────────────────────────────
# Set to explicit values to override auto-detection.
# Set to None to auto-detect from the data.
PANEL_B_MF_MIN = None       # e.g. 8.0
PANEL_B_MF_MAX = None       # e.g. 17.5
PANEL_B_DREF_MIN = None     # e.g. 1.6
PANEL_B_DREF_MAX = None     # e.g. 2.3

# ── Compounds to label in panel (a) ─────────────────────────────────────
# Edit this list to control which phases are labelled in the main band.
# Only compounds with MF < 6 (i.e. in the main perovskite band) are
# labelled; the list is ignored for compounds not in the data.
LABEL_COMPOUNDS_PANEL_A = [
    'MgSiO3', 'CaMnO3', 'CaTiO3', 'LaMnO3', 'LaFeO3',
    'NdNiO3', 'BaCeO3', 'SrCeO3', 'NaNbO3', 'NaTaO3',
    'SrRuO3', 'LaCrO3',
]

# ── Tier-to-marker mapping ──────────────────────────────────────────────
tier_map = {'Additive': 0, '1.1': 1, '2.1': 2, '2.2': 2,
            '2.3': 2, '2.4': 2, '2.5': 2, '3.1': 3}


def _plot_scatter_by_tier(ax, data, size_map, alpha_additive=0.5):
    """Plot scatter points with tier-based marker shapes and classification colours."""
    for tier in [0, 2, 3]:
        subset = data[data['tier'] == tier]
        if len(subset) == 0:
            continue
        if tier == 2:
            for code in ['2.2', '2.4', '2.5', '2.1']:
                sub = subset[subset['classification'] == code]
                if len(sub) == 0:
                    continue
                ax.scatter(sub['d_ref_ang'], sub['madelung_factor'],
                          c=CLASSIFICATION_COLOURS[code],
                          marker=TIER_MARKERS[tier], s=size_map.get(tier, 22),
                          alpha=0.75, edgecolors='white', linewidth=0.3,
                          zorder=3)
        else:
            code = 'Additive' if tier == 0 else '3.1'
            ax.scatter(subset['d_ref_ang'], subset['madelung_factor'],
                      c=CLASSIFICATION_COLOURS[code],
                      marker=TIER_MARKERS[tier], s=size_map.get(tier, 22),
                      alpha=alpha_additive if tier == 0 else 0.8,
                      edgecolors='white', linewidth=0.3,
                      zorder=2 if tier == 0 else 4)


def plot_mf_clustering(df_classified, save_path=None):
    """
    Two-panel Madelung factor vs d_ref scatter plot.

    Panel (a): all successfully decomposed structures, with common
    phases labelled at their median position.  Dashed box indicates
    the region shown in panel (b).

    Panel (b): zoomed view of the Z=1 / high-MF cluster.  Edge and
    outlier ICSD entries are labelled with compound + CollCode;
    clustered compositions (n >= 3) are labelled with entry count.

    The panel (b) window is auto-detected from the largest gap in the
    MF distribution above the main perovskite band, unless overridden
    by the PANEL_B_* variables.
    """
    ok = df_classified[df_classified['success'] == True].copy()
    ok['tier'] = ok['classification'].map(tier_map).fillna(-1).astype(int)
    ok['coll_code'] = ok['source_cif'].str.extract(r'CollCode(\d+)', expand=False)

    # ── Auto-detect panel (b) window ────────────────────────────────
    # Find the first significant gap (> 2.0) in MF above the main band
    mf_sorted = ok['madelung_factor'].sort_values().values
    all_gaps = np.diff(mf_sorted)
    mf_gap_lower = 5.0
    mf_gap_upper = 10.0
    for i in range(len(all_gaps)):
        if all_gaps[i] > 2.0 and mf_sorted[i] > 4.0:
            mf_gap_lower = mf_sorted[i]
            mf_gap_upper = mf_sorted[i + 1]
            break

    auto_mf_min = mf_gap_upper - 0.5

    # Exclude extreme outliers (e.g. SrRuO3 at MF~34)
    high_cluster = ok[(ok['madelung_factor'] >= auto_mf_min) &
                       (ok['madelung_factor'] < 20)]

    if len(high_cluster) > 0:
        auto_mf_max = high_cluster['madelung_factor'].max() + 0.5
        auto_dref_min = high_cluster['d_ref_ang'].min() - 0.05
        auto_dref_max = high_cluster['d_ref_ang'].max() + 0.05
    else:
        auto_mf_max = 18.0
        auto_dref_min = 1.6
        auto_dref_max = 2.3

    pb_mf_min = PANEL_B_MF_MIN if PANEL_B_MF_MIN is not None else auto_mf_min
    pb_mf_max = PANEL_B_MF_MAX if PANEL_B_MF_MAX is not None else auto_mf_max
    pb_dref_min = PANEL_B_DREF_MIN if PANEL_B_DREF_MIN is not None else auto_dref_min
    pb_dref_max = PANEL_B_DREF_MAX if PANEL_B_DREF_MAX is not None else auto_dref_max

    print(f"Panel (b) window: MF [{pb_mf_min:.1f}, {pb_mf_max:.1f}], "
          f"d_ref [{pb_dref_min:.2f}, {pb_dref_max:.2f}]")

    # ── Build figure ────────────────────────────────────────────────
    fig, (ax_a, ax_b) = plt.subplots(
        1, 2, figsize=(14, 6.5),
        gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.28}
    )

    # ── Panel (a): all structures ───────────────────────────────────
    _plot_scatter_by_tier(ax_a, ok, {0: 18, 2: 22, 3: 22})

    # Labels for common phases in the main band
    grouped_main = ok[ok['madelung_factor'] < 6].groupby('compound').agg(
        n=('compound', 'size'),
        mf_med=('madelung_factor', 'median'),
        dref_med=('d_ref_ang', 'median'),
    ).reset_index()

    texts_a = []
    for comp in LABEL_COMPOUNDS_PANEL_A:
        row = grouped_main[grouped_main['compound'] == comp]
        if len(row) == 0:
            continue
        r = row.iloc[0]
        n_str = f' ({int(r["n"])})' if r['n'] >= 5 else ''
        texts_a.append(ax_a.text(r['dref_med'], r['mf_med'],
                                 f'{comp}{n_str}',
                                 fontsize=7, color=SHEFFIELD['midnight'],
                                 fontstyle='italic'))

    # Label the SrRuO3 outlier at MF~34
    sr_out = ok[(ok['compound'] == 'SrRuO3') & (ok['madelung_factor'] > 30)]
    if len(sr_out) > 0:
        r = sr_out.iloc[0]
        texts_a.append(ax_a.text(r['d_ref_ang'], r['madelung_factor'],
                                 f'SrRuO3\nCollCode{r["coll_code"]}',
                                 fontsize=7, color=SHEFFIELD['midnight'],
                                 fontstyle='italic', ha='right'))

    adjust_text(texts_a, ax=ax_a,
                arrowprops=dict(arrowstyle='-', color='grey', alpha=0.4, lw=0.5),
                expand=(1.5, 1.5), force_text=(0.8, 0.8),
                ensure_inside_axes=True)

    # Gap reference line
    gap_y = (mf_gap_lower + mf_gap_upper) / 2
    ax_a.axhline(y=gap_y, color='grey', linestyle=':', linewidth=0.8, alpha=0.4)

    # Box showing panel (b) region
    rect = Rectangle((pb_dref_min, pb_mf_min),
                      pb_dref_max - pb_dref_min,
                      pb_mf_max - pb_mf_min,
                      linewidth=1.0, edgecolor=SHEFFIELD['dark_violet'],
                      facecolor='none', linestyle='--', alpha=0.6, zorder=5)
    ax_a.add_patch(rect)
    ax_a.text(pb_dref_max + 0.01, pb_mf_max, '(b)',
              fontsize=9, color=SHEFFIELD['dark_violet'], va='top',
              fontweight='bold')

    ax_a.set_xlabel('$d_\\mathrm{ref}$ / Å')
    ax_a.set_ylabel('Madelung factor')
    ax_a.set_title('(a)', loc='left', fontweight='bold')
    ax_a.spines['top'].set_visible(False)
    ax_a.spines['right'].set_visible(False)

    legend_elements = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor=SHEFFIELD['pastel_green'],
               markeredgecolor='white', markersize=7,
               label=r'Additive ($\Delta$ < 1%)'),
        Line2D([0], [0], marker='s', color='w',
               markerfacecolor=SHEFFIELD['coral'],
               markeredgecolor='white', markersize=7,
               label='2.1 Oxi state error'),
        Line2D([0], [0], marker='s', color='w',
               markerfacecolor=SHEFFIELD['peach'],
               markeredgecolor='white', markersize=7,
               label='2.2 Z = 1 artefact'),
        Line2D([0], [0], marker='s', color='w',
               markerfacecolor=SHEFFIELD['flamingo'],
               markeredgecolor='white', markersize=7,
               label='2.4 O non-stoich.'),
        Line2D([0], [0], marker='s', color='w',
               markerfacecolor=SHEFFIELD['brand_violet'],
               markeredgecolor='white', markersize=7,
               label='2.5 Wyckoff site error'),
        Line2D([0], [0], marker='D', color='w',
               markerfacecolor=SHEFFIELD['teal'],
               markeredgecolor='white', markersize=6,
               label='3.1 Genuine violation'),
    ]
    ax_a.legend(handles=legend_elements, frameon=False, fontsize=8,
                loc='upper left', handletextpad=0.5)

    # ── Panel (b): zoomed cluster ───────────────────────────────────
    cluster_b = ok[(ok['madelung_factor'] >= pb_mf_min) &
                   (ok['madelung_factor'] <= pb_mf_max) &
                   (ok['d_ref_ang'] >= pb_dref_min) &
                   (ok['d_ref_ang'] <= pb_dref_max)].copy()

    print(f"Panel (b): {len(cluster_b)} structures in window")
    _plot_scatter_by_tier(ax_b, cluster_b, {0: 35, 2: 40, 3: 40},
                          alpha_additive=0.6)

    # ── Label strategy for panel (b) ────────────────────────────────
    cluster_grouped = cluster_b.groupby('compound').agg(
        n=('compound', 'size'),
        mf_med=('madelung_factor', 'median'),
        dref_med=('d_ref_ang', 'median'),
        cls=('classification', 'first'),
        codes=('coll_code', lambda x: list(x)),
    ).reset_index()

    texts_b = []

    # Clustered compositions (n >= 3): label at median with count
    clustered = cluster_grouped[cluster_grouped['n'] >= 3]
    for _, r in clustered.iterrows():
        texts_b.append(ax_b.text(
            r['dref_med'], r['mf_med'],
            f'{r["compound"]} ({int(r["n"])})',
            fontsize=6.5, color=SHEFFIELD['midnight'],
            fontstyle='italic', fontweight='bold'))

    # Edge/outlier points: singletons and pairs at the edges
    if len(cluster_b) > 0:
        mf_hi = cluster_b['madelung_factor'].quantile(0.92)
        mf_lo = cluster_b['madelung_factor'].quantile(0.08)
        dref_hi = cluster_b['d_ref_ang'].quantile(0.92)
        dref_lo = cluster_b['d_ref_ang'].quantile(0.08)

        edge_candidates = cluster_grouped[cluster_grouped['n'] <= 2]
        for _, r in edge_candidates.iterrows():
            is_edge = (r['mf_med'] >= mf_hi or r['mf_med'] <= mf_lo or
                       r['dref_med'] >= dref_hi or r['dref_med'] <= dref_lo)
            if is_edge:
                cc = r['codes'][0] if r['codes'] else ''
                n_str = f' ({int(r["n"])})' if r['n'] == 2 else ''
                label = f'{r["compound"]}{n_str}\n{cc}' if cc else r['compound']
                texts_b.append(ax_b.text(
                    r['dref_med'], r['mf_med'], label,
                    fontsize=5.5, color=SHEFFIELD['midnight'],
                    fontstyle='italic'))

    if texts_b:
        adjust_text(texts_b, ax=ax_b,
                    arrowprops=dict(arrowstyle='-', color='grey',
                                    alpha=0.4, lw=0.5),
                    expand=(2.0, 2.0), force_text=(1.2, 1.2),
                    ensure_inside_axes=True)

    ax_b.set_xlabel('$d_\\mathrm{ref}$ / Å')
    ax_b.set_ylabel('Madelung factor')
    ax_b.set_title(f'(b) Z = 1 cluster detail  (n = {len(cluster_b)})',
                    loc='left', fontweight='bold')
    ax_b.spines['top'].set_visible(False)
    ax_b.spines['right'].set_visible(False)
    ax_b.set_xlim(pb_dref_min, pb_dref_max)
    ax_b.set_ylim(pb_mf_min, pb_mf_max)

    fig.subplots_adjust(wspace=0.28)

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()
    return fig


# ── Plot ─────────────────────────────────────────────────────────────────────
fig_mf = plot_mf_clustering(
    df_classified,
    save_path=MAPLE_DIR / 'fig2_mf_clustering.png'
)

## 6. Perovskite-Specific Analysis

Filter for perovskite space groups and examine MF clustering.

In [ ]:
# 6. Perovskite-Specific Analysis
# ─────────────────────────────────────────────────────────────────────────────

PEROVSKITE_SGS = ['Pnma', 'Pm-3m', 'R-3c', 'P4mm', 'I4/mcm',
                  'Cmcm', 'R3c', 'P2_1/n', 'Imma']

df_perov = df_classified[
    df_classified['space_group'].isin(PEROVSKITE_SGS)
].copy()

print(f"Perovskite candidates: {len(df_perov)} structures")
print(f"  Space groups: {df_perov['space_group'].value_counts().to_dict()}")
print(f"  Successfully decomposed: {df_perov['success'].sum()}")

# Classification breakdown for perovskites
print(f"\n  Classification breakdown:")
for code in ['Additive', '1.1', '2.1', '2.2', '2.3', '2.4', '3.1']:
    n = (df_perov['classification'] == code).sum()
    if n > 0:
        print(f"    {code:10s}  {n}")
print()

if len(df_perov) > 0:
    df_perov_ok = df_perov[df_perov['success']].sort_values('delta_pct')
    print("Perovskite additivity results (sorted by residual):")
    display(df_perov_ok[['compound', 'space_group', 'Z', 'maple_ternary',
                          'maple_binary_sum', 'delta_pct', 'classification',
                          'classification_label', 'madelung_factor']])



## 7. Pyrochlore-Specific Analysis

Filter for pyrochlore space groups (Fd-3m ordered, Fm-3m disordered).

In [ ]:
# 7. Pyrochlore-Specific Analysis
# ─────────────────────────────────────────────────────────────────────────────

PYROCHLORE_SGS = ['Fd-3m', 'Fm-3m']

df_pyro = df_classified[
    df_classified['space_group'].isin(PYROCHLORE_SGS)
].copy()

print(f"Fd-3m / Fm-3m candidates: {len(df_pyro)} structures")
print(f"  Space groups: {df_pyro['space_group'].value_counts().to_dict()}")
print(f"  Successfully decomposed: {df_pyro['success'].sum()}")

# Classification breakdown for pyrochlores
print(f"\n  Classification breakdown:")
for code in ['Additive', '1.1', '2.1', '2.2', '2.3', '2.4', '3.1']:
    n = (df_pyro['classification'] == code).sum()
    if n > 0:
        print(f"    {code:10s}  {n}")
print()

if len(df_pyro) > 0:
    df_pyro_ok = df_pyro[df_pyro['success']].sort_values('delta_pct')
    print("Pyrochlore/fluorite additivity results:")
    display(df_pyro_ok[['compound', 'space_group', 'Z', 'maple_ternary',
                         'maple_binary_sum', 'delta_pct', 'classification',
                         'classification_label', 'disordered',
                         'madelung_factor']])

    # ── Look for specific composition series ──────────────────────────
    ti_pyro = df_pyro_ok[df_pyro_ok['compound'].str.contains('Ti2',
                                                              na=False)]
    if len(ti_pyro) > 1:
        print(f"\nRE2Ti2O7 series ({len(ti_pyro)} entries):")
        display(ti_pyro[['compound', 'space_group', 'maple_ternary',
                          'delta_pct', 'madelung_factor', 'disordered',
                          'classification']])

    zr_pyro = df_pyro_ok[df_pyro_ok['compound'].str.contains('Zr',
                                                              na=False)]
    if len(zr_pyro) > 1:
        print(f"\nRE2Zr2O7 series ({len(zr_pyro)} entries):")
        display(zr_pyro[['compound', 'space_group', 'maple_ternary',
                          'delta_pct', 'madelung_factor', 'disordered',
                          'classification']])


## 8. Export Results

In [ ]:
# 8. Export Results
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = MAPLE_DIR

# Full classified table
out_csv = OUTPUT_DIR / 'additivity_classified.csv'
df_classified.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}  ({len(df_classified)} rows)")

# Verify the new columns are present in the export
expected_cols = ['classification', 'classification_tier', 'classification_label']
missing_cols = [c for c in expected_cols if c not in df_classified.columns]
if missing_cols:
    print(f"  WARNING: missing columns in export: {missing_cols}")
else:
    print(f"  Columns: {expected_cols} — all present")

# Perovskite subset
if len(df_perov) > 0:
    perov_csv = OUTPUT_DIR / 'additivity_perovskites.csv'
    df_perov.to_csv(perov_csv, index=False)
    print(f"Saved: {perov_csv}  ({len(df_perov)} rows)")

# Pyrochlore subset
if len(df_pyro) > 0:
    pyro_csv = OUTPUT_DIR / 'additivity_pyrochlores.csv'
    df_pyro.to_csv(pyro_csv, index=False)
    print(f"Saved: {pyro_csv}  ({len(df_pyro)} rows)")

# Binary reference table
binary_csv = OUTPUT_DIR / 'binary_oxide_reference.csv'
df_binary.to_csv(binary_csv, index=False)
print(f"Saved: {binary_csv}  ({len(df_binary)} rows)")

# Missing binaries summary
if df_additivity[~df_additivity['success']]['missing_binaries'].any():
    missing_series = (df_additivity[~df_additivity['success']]
                      ['missing_binaries']
                      .str.split('; ')
                      .explode()
                      .value_counts())
    print(f"\nMissing binary references (add these to the table):")
    print(missing_series.head(20))

# Classification summary for the export log
print(f"\nExport classification summary:")
print(f"  Total structures: {len(df_classified)}")
for tier_num, tier_name in {0: 'Additive', 1: 'Tier 1', 2: 'Tier 2',
                             3: 'Tier 3'}.items():
    n = (df_classified['classification_tier'] == tier_num).sum()
    if n > 0:
        print(f"  {tier_name:12s}  {n}")

print("\nDone.")


## 9. Notes and Limitations

### Binary reference MAPLE values

The fallback table uses approximate values that **must be replaced**
with MaplePy batch results on ICSD CIFs before publication.
Values marked "Estimated" are placeholders; prioritise computing
these from actual structures.

### Non-integer oxidation states

The decomposition function rounds oxidation states to the nearest
integer to determine the binary formula.  This is appropriate for
stoichiometric compounds but introduces ambiguity for mixed-valence
systems (*e.g.* Mn$^{3.5+}$ giving a mean that rounds to either
Mn$_2$O$_3$ or MnO$_2$).

### Oxygen non-stoichiometry

Hoppe’s additivity theorem applies strictly to stoichiometric
compounds.  Oxygen-deficient or oxygen-excess structures
(code 2.4) require modified decomposition that accounts
for vacancy or interstitial contributions.

### Z = 1 problem

Code 2.2 failures arise from pymatgen’s compositional reduction
algorithm rather than from the MAPLE calculation itself.  The
fix (extracting `_cell_formula_units_Z` from the CIF header) is
in progress in MaplePy and should be applied upstream.

In [ ]:
# 10. Additional Paper Figures
# ─────────────────────────────────────────────────────────────────────────────
#
# Figure 4: RE₂Ti₂O₇ titanate pyrochlore series
# Figure 5: RE₂Sn₂O₇ stannate pyrochlore series
# Figure 6: Lanthanide contraction in RE₂O₃ binary reference table
# Figure 7: Classification waterfall
#
# All figures use the Sheffield colour palette and style conventions
# established in earlier cells.
# ─────────────────────────────────────────────────────────────────────────────

import re as regex

# Shannon ionic radii (Å), CN = 8, oxidation state +3
# (pyrochlore A-site is 8-coordinate)
SHANNON_CN8 = {
    'La': 1.160, 'Ce': 1.143, 'Pr': 1.126, 'Nd': 1.109, 'Pm': 1.093,
    'Sm': 1.079, 'Eu': 1.066, 'Gd': 1.053, 'Tb': 1.040, 'Dy': 1.027,
    'Ho': 1.015, 'Er': 1.004, 'Tm': 0.994, 'Yb': 0.985, 'Lu': 0.977,
    'Y':  1.019,
}

# Shannon ionic radii (Å), CN = 6, oxidation state +3
# (used for binary RE₂O₃ figure)
SHANNON_CN6 = {
    'La': 1.032, 'Ce': 1.01,  'Pr': 0.99,  'Nd': 0.983, 'Pm': 0.97,
    'Sm': 0.958, 'Eu': 0.947, 'Gd': 0.938, 'Tb': 0.923, 'Dy': 0.912,
    'Ho': 0.901, 'Er': 0.89,  'Tm': 0.88,  'Yb': 0.868, 'Lu': 0.861,
}


def _build_pyrochlore_series(df_classified, pattern, shannon=SHANNON_CN8):
    """
    Extract median Δ and MF per RE element for a pyrochlore series.

    Uses only Additive + 3.1 entries (excludes disorder, Z=1, oxi-state
    artefacts).  Returns a DataFrame sorted by decreasing ionic radius.
    """
    rows = []
    for comp in sorted(df_classified['compound'].unique()):
        m = regex.match(pattern, comp)
        if not m:
            continue
        re_el = m.group(1)
        if re_el not in shannon:
            continue

        clean = df_classified[
            (df_classified['compound'] == comp) &
            (df_classified['classification'].isin(['Additive', '3.1']))
        ]
        if len(clean) == 0:
            continue

        all_entries = df_classified[df_classified['compound'] == comp]
        radius = shannon[re_el]

        rows.append({
            'element': re_el, 'compound': comp, 'radius': radius,
            'delta_median': clean['delta_pct'].median(),
            'delta_mean':   clean['delta_pct'].mean(),
            'delta_std':    clean['delta_pct'].std() if len(clean) > 1 else 0,
            'delta_min':    clean['delta_pct'].min(),
            'delta_max':    clean['delta_pct'].max(),
            'mf_median':    clean['madelung_factor'].median(),
            'n_clean':      len(clean),
            'n_total':      len(all_entries),
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    return df.sort_values('radius', ascending=False)


def plot_pyrochlore_series(df_classified, pattern, series_label,
                           fig_label='(a)', save_path=None):
    """
    Two-panel figure for a pyrochlore RE series:
      (a) Δ vs ionic radius — median ± range, with 1% threshold line
      (b) Madelung factor vs ionic radius
    """
    df = _build_pyrochlore_series(df_classified, pattern)
    if len(df) == 0:
        print(f"No data found for pattern {pattern}")
        return

    fig, (ax_a, ax_b) = plt.subplots(
        1, 2, figsize=(12, 5),
        gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.30}
    )

    radii = df['radius'].values
    delta_med = df['delta_median'].values
    delta_min = df['delta_min'].values
    delta_max = df['delta_max'].values
    mf_med = df['mf_median'].values
    elements = df['element'].values
    n_clean = df['n_clean'].values

    # ── Panel (a): Δ vs ionic radius ─────────────────────────────────
    # Error bars showing full range
    yerr_lo = delta_med - delta_min
    yerr_hi = delta_max - delta_med
    ax_a.errorbar(radii, delta_med, yerr=[yerr_lo, yerr_hi],
                  fmt='o', color=SHEFFIELD['brand_violet'],
                  markerfacecolor=SHEFFIELD['brand_violet'],
                  markeredgecolor='white', markersize=8,
                  ecolor=SHEFFIELD['powder_violet'], elinewidth=1.2,
                  capsize=3, capthick=1.0, zorder=3)

    # 1% threshold line
    ax_a.axhline(y=1.0, color=SHEFFIELD['coral'], linestyle='--',
                 linewidth=1.2, alpha=0.7, zorder=1)
    ax_a.text(ax_a.get_xlim()[0] + 0.003, 1.05, 'Hoppe 1%',
              fontsize=9, color=SHEFFIELD['coral'], va='bottom')

    # Label each point
    for el, r, d, n in zip(elements, radii, delta_med, n_clean):
        n_str = f' ({n})' if n > 1 else ''
        ax_a.annotate(f'{el}{n_str}', (r, d),
                      textcoords='offset points', xytext=(0, 10),
                      ha='center', fontsize=8, color=SHEFFIELD['midnight'])

    ax_a.set_xlabel('Shannon ionic radius (CN = 8, 3+) / Å')
    ax_a.set_ylabel(r'Additivity residual, $\Delta$ / %')
    ax_a.set_title(f'{fig_label} {series_label}', loc='left', fontweight='bold')
    ax_a.spines['top'].set_visible(False)
    ax_a.spines['right'].set_visible(False)
    ax_a.invert_xaxis()

    # ── Panel (b): MF vs ionic radius ────────────────────────────────
    ax_b.plot(radii, mf_med, 'o-',
              color=SHEFFIELD['teal'], markerfacecolor=SHEFFIELD['teal'],
              markeredgecolor='white', markersize=8, linewidth=1.5, zorder=3)

    for el, r, mf in zip(elements, radii, mf_med):
        ax_b.annotate(el, (r, mf), textcoords='offset points',
                      xytext=(0, 10), ha='center', fontsize=8,
                      color=SHEFFIELD['midnight'])

    ax_b.set_xlabel('Shannon ionic radius (CN = 8, 3+) / Å')
    ax_b.set_ylabel('Madelung factor')
    ax_b.set_title('(b) Madelung factor', loc='left', fontweight='bold')
    ax_b.spines['top'].set_visible(False)
    ax_b.spines['right'].set_visible(False)
    ax_b.invert_xaxis()

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()


def plot_lanthanide_contraction(binary_lookup, save_path=None):
    """
    Figure 6: Lanthanide contraction in RE₂O₃ binary MAPLE values.
    """
    lanthanides = ['La','Ce','Pr','Nd','Pm','Sm','Eu','Gd',
                   'Tb','Dy','Ho','Er','Tm','Yb','Lu']

    elements, radii, maples = [], [], []
    for el in lanthanides:
        formula = f"{el}2O3"
        if formula in binary_lookup and el in SHANNON_CN6:
            elements.append(el)
            radii.append(SHANNON_CN6[el])
            maples.append(binary_lookup[formula])

    if not elements:
        print("No RE₂O₃ entries found in binary reference table.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    # La₂O₃ is P-3m1, rest are Ia-3
    ia3_idx = [i for i, el in enumerate(elements) if el != 'La']
    la_idx  = [i for i, el in enumerate(elements) if el == 'La']

    # Plot Ia-3 series
    ia3_r = [radii[i] for i in ia3_idx]
    ia3_m = [maples[i] for i in ia3_idx]
    ax.plot(ia3_r, ia3_m, 'o-',
            color=SHEFFIELD['brand_violet'], markerfacecolor=SHEFFIELD['brand_violet'],
            markeredgecolor='white', markersize=8, linewidth=1.5, zorder=3)

    # Linear fit to Ia-3 entries
    coeffs = np.polyfit(ia3_r, ia3_m, 1)
    fit_r = np.linspace(min(ia3_r) - 0.02, max(ia3_r) + 0.02, 100)
    fit_m = np.polyval(coeffs, fit_r)
    ax.plot(fit_r, fit_m, '--', color=SHEFFIELD['powder_violet'],
            linewidth=1.0, alpha=0.7, zorder=1)

    # Plot La separately (different structure type)
    if la_idx:
        i = la_idx[0]
        ax.plot(radii[i], maples[i], 's',
                color=SHEFFIELD['coral'], markeredgecolor='white',
                markersize=10, zorder=4)
        ax.annotate(r'$P\overline{3}m1$', (radii[i], maples[i]),
                    textcoords='offset points', xytext=(-20, -18),
                    fontsize=8, color=SHEFFIELD['coral'], fontstyle='italic',
                    arrowprops=dict(arrowstyle='-', color=SHEFFIELD['coral'],
                                    lw=0.8, alpha=0.6))

    # Label each point
    for el, r, m in zip(elements, radii, maples):
        idx = elements.index(el)
        va = 'bottom' if idx % 2 == 0 else 'top'
        offset = 12 if va == 'bottom' else -12
        ax.annotate(el, (r, m), textcoords='offset points',
                    xytext=(0, offset), ha='center', va=va,
                    fontsize=8, color=SHEFFIELD['midnight'])

    slope = coeffs[0]
    ax.text(0.03, 0.95,
            f'$Ia\\overline{{3}}$ fit: slope = {slope:.0f}'
            r' kcal mol$^{-1}$ Å$^{-1}$',
            transform=ax.transAxes, fontsize=9,
            color=SHEFFIELD['powder_violet'], va='top')

    ax.set_xlabel('Shannon ionic radius (CN = 6, 3+) / Å')
    ax.set_ylabel(r'MAPLE(RE$_2$O$_3$) / kcal mol$^{-1}$')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_xaxis()

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()


def plot_classification_waterfall(df_classified, save_path=None):
    """
    Figure 7: Classification waterfall — horizontal stacked bar showing
    how structures are sorted into classification tiers.
    """
    code_order = ['Additive', '1.1', '2.1', '2.2', '2.3', '2.4', '2.5', '3.1']
    code_labels = {
        'Additive': 'Additive\n(Δ < 1%)',
        '1.1': '1.1\nMissing\nbinary ref.',
        '2.1': '2.1\nOxi state\nerror',
        '2.2': '2.2\nZ = 1\nartefact',
        '2.3': '2.3\nDisorder /\nmean-field',
        '2.4': '2.4\nO non-\nstoich.',
        '2.5': '2.5\nStructural\ncoord. error',
        '3.1': '3.1\nGenuine\nviolation',
    }
    code_colours = {
        'Additive': SHEFFIELD['pastel_green'],
        '1.1':      SHEFFIELD['pale_violet'],
        '2.1':      SHEFFIELD['coral'],
        '2.2':      SHEFFIELD['peach'],
        '2.3':      SHEFFIELD['powder_violet'],
        '2.4':      SHEFFIELD['flamingo'],
        '2.5':      SHEFFIELD['brand_violet'],
        '3.1':      SHEFFIELD['teal'],
    }

    counts = []
    codes_present = []
    for code in code_order:
        n = (df_classified['classification'] == code).sum()
        if n > 0:
            counts.append(n)
            codes_present.append(code)

    total = sum(counts)

    fig, ax = plt.subplots(figsize=(10, 3.5))

    left = 0
    for code, count in zip(codes_present, counts):
        ax.barh(0, count, left=left, height=0.6,
                color=code_colours[code],
                edgecolor='white', linewidth=1.5)

        if count > 0:
            pct = count / total * 100
            cx = left + count / 2
            dark_codes = {'Additive', '2.2', '2.4', '1.1', '2.3'}
            text_colour = SHEFFIELD['midnight'] if code in dark_codes else 'white'

            if count >= 30:
                ax.text(cx, 0, f'{count}\n({pct:.0f}%)',
                        ha='center', va='center', fontsize=9,
                        fontweight='bold', color=text_colour)
            elif count >= 10:
                ax.text(cx, 0, f'{count}',
                        ha='center', va='center', fontsize=8,
                        fontweight='bold', color=text_colour)
            else:
                ax.text(cx, 0.45, f'{count}',
                        ha='center', va='bottom', fontsize=8,
                        color=code_colours[code], fontweight='bold')

        left += count

    # Category labels below — code only, staggered with leader lines
    left = 0
    for i, (code, count) in enumerate(zip(codes_present, counts)):
        if count > 0:
            cx = left + count / 2
            y_pos = -0.38 if i % 2 == 0 else -0.58
            label = code if code != 'Additive' else 'Add.'
            ax.annotate(label, xy=(cx, -0.30), xytext=(cx, y_pos),
                        ha='center', va='top', fontsize=8,
                        fontweight='bold', color=SHEFFIELD['midnight'],
                        arrowprops=dict(arrowstyle='-', color='grey',
                                        lw=0.6, alpha=0.5))
        left += count

    # Tier brackets above
    tier_groups = {}
    left = 0
    for code, count in zip(codes_present, counts):
        tier = 0 if code == 'Additive' else int(code[0])
        tier_name = f'Tier {tier}'
        if tier_name not in tier_groups:
            tier_groups[tier_name] = [left, left + count]
        else:
            tier_groups[tier_name][1] = left + count
        left += count

    tier_colours = {
        'Tier 0': SHEFFIELD['peak_green'],
        'Tier 1': SHEFFIELD['powder_violet'],
        'Tier 2': SHEFFIELD['coral'],
        'Tier 3': SHEFFIELD['teal'],
    }

    for tier_name, (x0, x1) in tier_groups.items():
        width = x1 - x0
        if width > 15:
            mid = (x0 + x1) / 2
            colour = tier_colours.get(tier_name, SHEFFIELD['midnight'])
            ax.annotate('', xy=(x0, 0.48), xytext=(x1, 0.48),
                        arrowprops=dict(arrowstyle='|-|', color=colour,
                                        lw=1.2, mutation_scale=4))
            ax.text(mid, 0.58, tier_name, ha='center', va='bottom',
                    fontsize=8, color=colour, fontweight='bold')

    ax.set_xlim(-5, total + 5)
    ax.set_ylim(-1.0, 1.0)
    ax.set_xlabel(f'Number of structures (total = {total})')
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()


# ── Generate Figures ─────────────────────────────────────────────────────────

# Figure 4: Titanate pyrochlore series
plot_pyrochlore_series(
    df_classified,
    pattern=r'^([A-Z][a-z]?)2Ti2O7$',
    series_label=r'RE$_2$Ti$_2$O$_7$ titanate series',
    fig_label='(a)',
    save_path=MAPLE_DIR / 'fig4_titanate_series.png'
)

# Figure 5: Stannate pyrochlore series
plot_pyrochlore_series(
    df_classified,
    pattern=r'^([A-Z][a-z]?)2Sn2O7$',
    series_label=r'RE$_2$Sn$_2$O$_7$ stannate series',
    fig_label='(a)',
    save_path=MAPLE_DIR / 'fig5_stannate_series.png'
)

# Figure 6: Lanthanide contraction in binary RE₂O₃
binary_lookup_fig6 = dict(zip(df_binary['formula'], df_binary['maple_kcal_mol']))
plot_lanthanide_contraction(
    binary_lookup_fig6,
    save_path=MAPLE_DIR / 'fig6_lanthanide_contraction.png'
)

# Figure 7: Classification waterfall
plot_classification_waterfall(
    df_classified,
    save_path=MAPLE_DIR / 'fig7_classification_waterfall.png'
)